In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [49]:
pd.set_option('display.max_columns', None)   
pd.set_option('display.max_rows', None)   

## Importing the data and merging it

In [2]:
data = pd.read_csv('../raw_data/secom.data', header=None, delimiter=" ")
labels = pd.read_csv('../raw_data/secom_labels.data', header=None, delimiter=" ")

In [3]:
data.shape

(1567, 590)

In [4]:
data.head()

,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [5]:
labels.head()

,0,1
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00


In [6]:
labels.shape

(1567, 2)

### Rename label columns 

In [7]:
labels = labels.rename({0: "Label", 1: "timestamp"}, axis='columns')

In [8]:
df = data.merge(labels, left_index=True, right_index=True)

In [9]:
df.shape

(1567, 592)

In [10]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,582,583,584,585,586,587,588,589,Label,timestamp
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1,19/07/2008 11:55:00
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1,19/07/2008 12:32:00
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1,19/07/2008 13:17:00
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1,19/07/2008 14:43:00
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1,19/07/2008 15:22:00


## Phase 2: Initial Data Inspection

#### Count rows, columns, pass examples, fail examples, and failure percentage.

In [11]:
# Rows and Columns count
print(f'Amount of rows {df.shape[0]}')
print(f'Amount of columns {df.shape[1]}')

Amount of rows 1567
Amount of columns 592


In [12]:
pass_fail = df.Label.value_counts()

In [13]:
df.Label.value_counts(normalize=True)

Label
-1    0.933631
 1    0.066369
Name: proportion, dtype: float64

In [14]:
print(f'Pass example count {pass_fail[-1]} that is 93.3631% of the total')
print(f'Fail example count {pass_fail[1]} that is the 6.6369% of the total')

Pass example count 1463 that is 93.3631% of the total
Fail example count 104 that is the 6.6369% of the total


### Checking for null values

In [15]:
df.isnull().sum()

0             6
1             7
2            14
3            14
4            14
             ..
587           1
588           1
589           1
Label         0
timestamp     0
Length: 592, dtype: int64

In [16]:
df.isnull().sum().sum()

np.int64(41951)

### Identify constant and near-constant features.

>Constant features were identified as columns with one or fewer unique non-missing values. Near-constant features were identified as columns where one observed value accounted for at least 98% of the non-missing observations. These features were reviewed because they provide little variation for distinguishing pass and fail examples.

In [17]:
def constant_near_constant_report(
    X: pd.DataFrame,
    near_constant_threshold: float = 0.98
) -> pd.DataFrame:
    """
    Identify constant and near-constant features in a dataframe.

    Constant feature:
        A column with 0 or 1 unique non-missing values.

    Near-constant feature:
        A column where one value dominates at least near_constant_threshold
        of the non-missing observations.

    Parameters
    ----------
    X:
        Feature dataframe.
    near_constant_threshold:
        Dominance threshold. 0.98 means one value appears in >= 98%
        of non-missing rows.

    Returns
    -------
    report:
        DataFrame with one row per feature and flags for constant/near-constant.
    """

    rows = []

    for col in X.columns:
        s = X[col]
        n_rows = len(s)
        n_missing = s.isna().sum()
        missing_rate = n_missing / n_rows

        non_missing = s.dropna()
        n_non_missing = len(non_missing)

        # Unique values excluding missing values
        n_unique_non_missing = non_missing.nunique()

        # Unique values including NaN as a value
        n_unique_including_missing = s.nunique(dropna=False)

        if n_non_missing == 0:
            most_common_value = np.nan
            most_common_count = 0
            dominance_rate_non_missing = np.nan
            dominance_rate_total = np.nan
        else:
            counts = non_missing.value_counts(dropna=True)
            most_common_value = counts.index[0]
            most_common_count = counts.iloc[0]

            # Among observed values only
            dominance_rate_non_missing = most_common_count / n_non_missing

            # Among all rows, including missing rows in the denominator
            dominance_rate_total = most_common_count / n_rows

        is_constant = n_unique_non_missing <= 1

        is_near_constant = (
            not is_constant
            and dominance_rate_non_missing >= near_constant_threshold
        )

        rows.append({
            "feature": col,
            "n_rows": n_rows,
            "n_missing": n_missing,
            "missing_rate": missing_rate,
            "n_unique_non_missing": n_unique_non_missing,
            "n_unique_including_missing": n_unique_including_missing,
            "most_common_value": most_common_value,
            "most_common_count": most_common_count,
            "dominance_rate_non_missing": dominance_rate_non_missing,
            "dominance_rate_total": dominance_rate_total,
            "is_constant": is_constant,
            "is_near_constant": is_near_constant,
        })

    report = pd.DataFrame(rows)

    return report.sort_values(
        by=["is_constant", "is_near_constant", "dominance_rate_non_missing"],
        ascending=[False, False, False]
    )

In [18]:
report = constant_near_constant_report(df.drop(columns=["Label"]), near_constant_threshold=0.98)

In [19]:
report

,feature,n_rows,n_missing,missing_rate,n_unique_non_missing,n_unique_including_missing,most_common_value,most_common_count,dominance_rate_non_missing,dominance_rate_total,is_constant,is_near_constant
5,5,1567,14,0.008934,1,2,100.0,1553,1.000000,0.991066,True,False
13,13,1567,3,0.001914,1,2,0.0,1564,1.000000,0.998086,True,False
42,42,1567,1,0.000638,1,2,70.0,1566,1.000000,0.999362,True,False
49,49,1567,1,0.000638,1,2,1.0,1566,1.000000,0.999362,True,False
52,52,1567,1,0.000638,1,2,0.0,1566,1.000000,0.999362,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
527,527,1567,0,0.000000,1549,1549,6.9889,2,0.001276,0.001276,False,False
363,363,1567,51,0.032546,1516,1517,560.2658,1,0.000660,0.000638,False,False
472,472,1567,7,0.004467,1560,1561,63.7987,1,0.000641,0.000638,False,False
296,296,1567,2,0.001276,1565,1566,496.1582,1,0.000639,0.000638,False,False


In [20]:
constant_features = report.loc[report["is_constant"], "feature"].tolist()
near_constant_features = report.loc[report["is_near_constant"], "feature"].tolist()

In [21]:
len(near_constant_features)

10

In [22]:
len(constant_features)

116

In [23]:
for threshold in [0.95, 0.98, 0.99]:
    report = constant_near_constant_report(df.drop(columns=["Label"]), near_constant_threshold=threshold)
    print(
        threshold,
        "constant:",
        report["is_constant"].sum(),
        "near-constant:",
        report["is_near_constant"].sum()
    )

0.95 constant: 116 near-constant: 10
0.98 constant: 116 near-constant: 10
0.99 constant: 116 near-constant: 6


## Phase 3: Missingness as Process Information

### Check duplicate rows and feature types

In [24]:
data.duplicated().sum()

np.int64(0)

In [25]:
df.dtypes.value_counts()

float64    590
int64        1
str          1
Name: count, dtype: int64

In [26]:
print(f'Label column is integer (-1, 1) \nTimestamp column string (needs to be converted to datetime)\nThe rest of the columns are float data type')

Label column is integer (-1, 1) 
Timestamp column string (needs to be converted to datetime)
The rest of the columns are float data type


In [27]:
df.timestamp = pd.to_datetime(df.timestamp, format="%d/%m/%Y %H:%M:%S")

In [28]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,582,583,584,585,586,587,588,589,Label,timestamp
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1,2008-07-19 11:55:00
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1,2008-07-19 12:32:00
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1,2008-07-19 13:17:00
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1,2008-07-19 14:43:00
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1,2008-07-19 15:22:00


#### Calculate missing percentage per feature.

In [29]:
col_50_pre_null_val = df.columns[df.isnull().mean() > 0.5]

In [30]:
len(col_50_pre_null_val)

28

#### Compare missingness rates between pass and fail groups.

In [46]:
def missingness_by_target_report(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """
    Compare missing-value patterns between pass and fail groups.

    This function is useful for the SECOM dataset because missing values may not
    be random. In semiconductor manufacturing data, missingness can sometimes
    reflect skipped measurements, route differences, sensor/tool availability,
    metrology issues, or abnormal process behavior.

    SECOM target definition:
        -1 = pass
         1 = fail

    Parameters
    ----------
    X : pd.DataFrame
        Feature dataframe. This should contain only process/sensor features.
        Do not include the target column or timestamp column here.

    y : pd.Series
        Target labels aligned with X.
        Expected values:
            -1 = pass
             1 = fail

    Returns
    -------
    pd.DataFrame
        One row per feature with missingness statistics for pass and fail groups.
    """

    # -------------------------------------------------------------------------
    # Safety check:
    # X and y must have the same index so the Boolean masks line up correctly.
    # If the index does not match, the pass/fail filtering may be wrong.
    # -------------------------------------------------------------------------
    assert X.index.equals(y.index), "X and y indexes do not match."

    # Make a copy of y so the function does not accidentally modify the original.
    y = y.copy()

    # -------------------------------------------------------------------------
    # Create Boolean masks for the two target groups.
    #
    # fail_mask is True for rows where the production entity failed.
    # pass_mask is True for rows where the production entity passed.
    # -------------------------------------------------------------------------
    fail_mask = y == 1
    pass_mask = y == -1

    # Count how many passing and failing examples exist.
    # For SECOM, you should expect about 104 failures.
    n_pass = pass_mask.sum()
    n_fail = fail_mask.sum()

    # Optional safety check:
    # Make sure both classes exist before calculating rates.
    if n_pass == 0 or n_fail == 0:
        raise ValueError("Both pass (-1) and fail (1) classes must exist in y.")

    # This list will store one dictionary of results per feature.
    rows = []

    # -------------------------------------------------------------------------
    # Loop through every feature column.
    # For each feature, we calculate how often it is missing in pass rows
    # and how often it is missing in fail rows.
    # -------------------------------------------------------------------------
    for col in X.columns:

        # Boolean Series:
        # True means the value is missing for this feature in that row.
        # False means the value is present.
        missing = X[col].isna()

        # Count missing values among passing examples only.
        pass_missing_count = missing[pass_mask].sum()

        # Count missing values among failing examples only.
        fail_missing_count = missing[fail_mask].sum()

        # Count total missing values for this feature across all rows.
        total_missing_count = missing.sum()

        # Missing rate among passing examples.
        # Example: 100 missing pass rows / 1463 pass rows = 0.068
        pass_missing_rate = pass_missing_count / n_pass

        # Missing rate among failing examples.
        # Example: 30 missing fail rows / 104 fail rows = 0.288
        fail_missing_rate = fail_missing_count / n_fail

        # Difference between fail missing rate and pass missing rate.
        #
        # Positive value:
        #     Feature is missing more often in failed examples.
        #
        # Negative value:
        #     Feature is missing more often in passing examples.
        #
        # Near zero:
        #     Missingness is similar between pass and fail groups.
        missing_rate_diff = fail_missing_rate - pass_missing_rate

        # ---------------------------------------------------------------------
        # Among all rows where this feature is missing,
        # calculate what percentage belong to the failure class.
        #
        # This is useful because SECOM failures are rare.
        # If the general failure rate is about 6.6%, but 40% of rows with this
        # feature missing are failures, then missingness may be informative.
        # ---------------------------------------------------------------------
        if total_missing_count > 0:
            fail_share_among_missing = fail_missing_count / total_missing_count
        else:
            # If the feature is never missing, this value is not applicable.
            fail_share_among_missing = np.nan

        # ---------------------------------------------------------------------
        # Convert Boolean masks to 0/1 so we can compare missingness directly
        # against the failure flag.
        #
        # missing_as_int:
        #     1 = this feature is missing in this row
        #     0 = this feature is not missing
        #
        # fail_as_int:
        #     1 = this row is a failure
        #     0 = this row is not a failure
        # ---------------------------------------------------------------------
        missing_as_int = missing.astype(int)
        fail_as_int = fail_mask.astype(int)

        # Check whether this feature's missingness pattern exactly matches
        # the failure flag.
        #
        # True means:
        #     feature is missing for every failed row
        #     feature is not missing for every passing row
        #
        # This is very suspicious and should be reviewed for possible leakage
        # or data-generation artifacts before using the feature in a model.
        exact_match_fail_flag = missing_as_int.equals(fail_as_int)

        # Check whether every failed example has this feature missing.
        #
        # Important:
        # This does not mean the feature is missing only for failures.
        # It may also be missing for some passing rows.
        all_fails_missing = fail_missing_count == n_fail

        # Check whether missingness occurs only in failed examples.
        #
        # True means:
        #     no passing rows are missing this feature
        #     at least one failed row is missing this feature
        #
        # This can be a strong rare-failure signal, but it still needs review.
        missing_only_when_fail = (
            total_missing_count > 0
            and pass_missing_count == 0
            and fail_missing_count > 0
        )

        # Check whether missingness perfectly separates failures from passes.
        #
        # True means:
        #     all failed rows are missing this feature
        #     no passing rows are missing this feature
        #
        # This is equivalent to a perfect missingness-based failure flag.
        # In real projects, this should be treated carefully because it may
        # indicate leakage, a downstream measurement, or a data artifact.
        perfect_failure_missing_signal = (
            fail_missing_count == n_fail
            and pass_missing_count == 0
        )

        # Store all calculated values for this feature.
        rows.append({
            "feature": col,

            # Dataset-level class counts
            "n_pass": n_pass,
            "n_fail": n_fail,

            # Missing-value counts
            "pass_missing_count": pass_missing_count,
            "fail_missing_count": fail_missing_count,
            "total_missing_count": total_missing_count,

            # Missing-value rates
            "pass_missing_rate": pass_missing_rate,
            "fail_missing_rate": fail_missing_rate,
            "missing_rate_diff_fail_minus_pass": missing_rate_diff,

            # Among missing rows, how many are failures?
            "fail_share_among_missing": fail_share_among_missing,

            # Special missingness pattern flags
            "all_fails_missing": all_fails_missing,
            "missing_only_when_fail": missing_only_when_fail,
            "perfect_failure_missing_signal": perfect_failure_missing_signal,
            "exact_match_fail_flag": exact_match_fail_flag,
        })

    # Convert the list of dictionaries into a dataframe.
    report = pd.DataFrame(rows)

    # -------------------------------------------------------------------------
    # Sort the report so the most important/suspicious features appear first.
    #
    # Priority 1:
    #     Features where missingness perfectly matches the failure label.
    #
    # Priority 2:
    #     Features missing much more often in failures than passes.
    #
    # Priority 3:
    #     Features missing in more failed rows.
    # -------------------------------------------------------------------------
    report = report.sort_values(
        by=[
            "perfect_failure_missing_signal",
            "missing_rate_diff_fail_minus_pass",
            "fail_missing_count",
        ],
        ascending=[False, False, False]
    )

    return report

In [47]:
missingness_target_report = missingness_by_target_report(df.drop(columns='Label'), df.Label)

In [52]:
missingness_target_report.head()

,feature,n_pass,n_fail,pass_missing_count,fail_missing_count,total_missing_count,pass_missing_rate,fail_missing_rate,missing_rate_diff_fail_minus_pass,fail_share_among_missing,all_fails_missing,missing_only_when_fail,perfect_failure_missing_signal,exact_match_fail_flag
109,109,1463,104,944,74,1018,0.645249,0.711538,0.066289,0.072692,False,False,False,False
110,110,1463,104,944,74,1018,0.645249,0.711538,0.066289,0.072692,False,False,False,False
111,111,1463,104,944,74,1018,0.645249,0.711538,0.066289,0.072692,False,False,False,False
244,244,1463,104,944,74,1018,0.645249,0.711538,0.066289,0.072692,False,False,False,False
245,245,1463,104,944,74,1018,0.645249,0.711538,0.066289,0.072692,False,False,False,False


> No desicion about the missigness will be made at this time

## Phase 4: Target Imbalance and Manufacturing Risk